# Day 2 Group Exercise: Data Quality Check

In-class group notebook. Focus on inspection, KPI measurement, validation, and reporting.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
repo_root = Path.cwd().resolve().parents[1]
df = pd.read_csv(repo_root / 'data' / 'raw' / 'week2_group_exercise_orders_messy.csv')
print('Shape:', df.shape)
df.head(10)


## Group tasks

1. inspect the dataset
2. identify major data quality issues
3. map issues to quality dimensions
4. calculate at least 3 KPIs
5. define 3 validation rules
6. summarize findings
7. suggest cleaning actions for next week


## Step 1 - Inspect dataset


In [ ]:
df.info()

# Add extra checks here


### Write your observations here

- Observation 1
- Observation 2
- Observation 3


## Step 2 - Map issues to dimensions


In [ ]:
template = pd.DataFrame([['Example: missing customer_id','Completeness','Medium','Impacts customer segmentation']], columns=['Issue','Dimension','Severity','Why it matters'])
template


### Add your mapping here

Write your issue-to-dimension analysis here.


## Step 3 - Calculate at least 3 KPIs


In [ ]:
total_cells=df.shape[0]*df.shape[1]
completeness=1-(df.isna().sum().sum()/total_cells)
dup_rate=df.duplicated(subset=['order_id']).mean()
order_dt=pd.to_datetime(df['order_date'], errors='coerce', format='mixed')
ship_dt=pd.to_datetime(df['ship_date'], errors='coerce', format='mixed')
valid=order_dt.notna() & ship_dt.notna()
on_time=valid & (ship_dt>=order_dt) & ((ship_dt-order_dt).dt.days<=10)
timeliness=on_time.sum()/valid.sum() if valid.sum() else np.nan
pd.DataFrame({'KPI':['Completeness','Duplication(order_id)','Timeliness(<=10d)'],'Value':[completeness,dup_rate,timeliness]})


### Add your KPI summary here

- KPI 1
- KPI 2
- KPI 3
- Interpretation


## Step 4 - Define 3 validation rules


In [ ]:
rules={
'order_id_required': df['order_id'].isna() | (df['order_id'].astype(str).str.strip()==''),
'order_amount_non_negative': pd.to_numeric(df['order_amount'], errors='coerce')<0,
'quantity_positive': pd.to_numeric(df['quantity'], errors='coerce')<=0,
}
pd.DataFrame({k:int(v.sum()) for k,v in rules.items()}, index=['affected_rows']).T


### Which issue should be prioritized first and why?

Write your answer here.


## Optional helper function template (for teams)

Use this template if your team wants cleaner, reusable code.
Try changing parameter values to test different assumptions.


In [ ]:
def flag_timeliness_issues(order_dates, ship_dates, max_days=10):
    """Return a boolean mask for timeliness violations.

    Parameters
    ----------
    order_dates : pd.Series (datetime)
        Parsed order creation date.
    ship_dates : pd.Series (datetime)
        Parsed shipping date.
    max_days : int, default=10
        Maximum acceptable shipping delay in days.

    Returns
    -------
    pd.Series (bool)
        True where timeliness rule is violated.
    """
    valid = order_dates.notna() & ship_dates.notna()
    delay = (ship_dates - order_dates).dt.days
    violation = (~valid) | (delay < 0) | (delay > max_days)
    return violation

# Example usage:
order_dt = pd.to_datetime(df['order_date'], errors='coerce', format='mixed')
ship_dt = pd.to_datetime(df['ship_date'], errors='coerce', format='mixed')
late_or_invalid = flag_timeliness_issues(order_dt, ship_dt, max_days=10)
print('Timeliness violations:', int(late_or_invalid.sum()))


## Final group summary

Write your final audit-style summary here (key issues, KPI highlights, next-step recommendations).
